### Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

Github page for Open Web Search: https://github.com/Aas-ee/open-webSearch

In [1]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters

Note: you may need to restart the kernel to use updated packages.


In [2]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

aca_mcp_server_open_web_search_fqdn = ! terraform -chdir=infra output -raw aca_mcp_server_open_web_search_fqdn
aca_mcp_server_open_web_search_fqdn = aca_mcp_server_open_web_search_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_open_web_search_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
MCP Server Endpoint: aca-mcp-server-open-web-search.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


In [36]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Connect to the LLM endpoint hosted on ACA with GPU
# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 4096 # 8736 # 131072 # 512
# )

# Connect to the LLM endpoint hosted on Foundry
model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    # max_completion_tokens=512
)

In [37]:
response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- math and reasoning
- planning and organization

A few useful things to know about me:
- I don’t have feelings, beliefs, or personal experiences.
- I generate responses based on patterns in data I was trained on.
- I can be very helpful, but I can also make mistakes, so important facts should be verified.
- I don’t automatically know real-time information unless it’s provided to me or I have access to tools that supply it.

If you want, I can also tell you about:
- my strengths and limitations
- how I “think” at a high level
- what I’m good at compared with search engines
- how to get better answers from me

### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [13]:
# Invoke a specific tool
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke({"query": "What are the latest LLM models?", "limit": 20, "engines": ["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "What are the latest LLM models?",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 20,
  "results": [
    {
      "title": "LLM Leaderboard 2026: Compare 300+ Top AI Models by Intelligence, Speed ...",
      "url": "https://llm-stats.com/",
      "description": "<b>The</b> <b>LLM</b> Leaderboard \u2014 independent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI <b>models</b> by intelligence, speed and price. Composite <b>LLM</b> Stats Score updated continuously from public benchmarks and live API metrics.",
      "source": "llm-stats.com",
      "engine": "duckduckgo"
    },
    {
      "title": "LLM Leaderboard - Comparison of over 100 AI models from OpenAI, Google ...",
      "url": "https://artificialanalysis.ai/leaderboards/models",
      "description": "Comparison and ranking the performance of over 100 AI <b>models</b> (<b>LLMs</b>) across key metrics including intelligence, price, performance and speed (output speed - tokens per second &amp; laten

In [14]:
# Extract first URL from search results
first_url = data["results"][0]["url"]
print("First URL:", first_url)

# Find the tool called "search" by name
fetch_web_content_tool = next(t for t in mcp_tools_web_search if t.name == "fetchWebContent")

# Fetch the page content using the fetch tool
result = await fetch_web_content_tool.ainvoke({"url": first_url, "maxChars": 30000})

# Parse and display the fetched content
web_content_json = json.loads(result[0]["text"])
print(json.dumps(web_content_json, indent=2))

First URL: https://llm-stats.com/
{
  "url": "https://llm-stats.com/",
  "finalUrl": "https://llm-stats.com/",
  "contentType": "text/html; charset=utf-8",
  "title": "LLM Leaderboard 2026: Compare 300+ Top AI Models by Intelligence, Speed & Price",
  "retrievalMethod": "request",
  "truncated": false,
  "content": "AI LeaderboardsLLM LeaderboardOpen LLM LeaderboardCodingWritingMathResearchLong ContextTool CallingReasoningMoreLLM Leaderboard \u2014 Compare 300+ Top AI Models by Intelligence, Speed & PriceIndependent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI models \u2014 composite LLM Stats Score, updated continuously from public benchmarks and live API metrics.Current leadersLiveClaude Mythos Previewleads on reasoning94.6%gpqaGemini 3.1 Prowins at coding21arenaKimi K2.6cheapest in the top 10$0.95/M tokMercury 2fastest output1197tok/sGrok-4.20 Beta Non-Reasoninglongest context window2.0M tokenstokensKimi K2.6best open-weights90.5%gpqaLLMLLMImage GenerationImageVideo G

In [15]:
from markdownify import markdownify

markdownify(html=web_content_json["content"])

'AI LeaderboardsLLM LeaderboardOpen LLM LeaderboardCodingWritingMathResearchLong ContextTool CallingReasoningMoreLLM Leaderboard — Compare 300+ Top AI Models by Intelligence, Speed & PriceIndependent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI models — composite LLM Stats Score, updated continuously from public benchmarks and live API metrics.Current leadersLiveClaude Mythos Previewleads on reasoning94.6%gpqaGemini 3.1 Prowins at coding21arenaKimi K2.6cheapest in the top 10$0.95/M tokMercury 2fastest output1197tok/sGrok-4.20 Beta Non-Reasoninglongest context window2.0M tokenstokensKimi K2.6best open-weights90.5%gpqaLLMLLMImage GenerationImageVideo GenerationVideoText-to-SpeechTTSSpeech-to-TextSTTEmbeddingsEmbeddingsFull leaderboardFull298OpenAll90d30dRankModelLLM StatsReasoningCodingAgentCode ArenaContextSpeedPricing $/MLicense1Claude Mythos PreviewUNRELEASEDAnthropic70.371.357.348.9————Proprietary2GPT-5.5OpenAI64.362.953.144.11,8471.1M28c/s$7.78Proprietary3Claude Opus 

In [16]:
from IPython.display import display, Markdown

display(Markdown(web_content_json["content"]))  # if fetch_data is a string

AI LeaderboardsLLM LeaderboardOpen LLM LeaderboardCodingWritingMathResearchLong ContextTool CallingReasoningMoreLLM Leaderboard — Compare 300+ Top AI Models by Intelligence, Speed & PriceIndependent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI models — composite LLM Stats Score, updated continuously from public benchmarks and live API metrics.Current leadersLiveClaude Mythos Previewleads on reasoning94.6%gpqaGemini 3.1 Prowins at coding21arenaKimi K2.6cheapest in the top 10$0.95/M tokMercury 2fastest output1197tok/sGrok-4.20 Beta Non-Reasoninglongest context window2.0M tokenstokensKimi K2.6best open-weights90.5%gpqaLLMLLMImage GenerationImageVideo GenerationVideoText-to-SpeechTTSSpeech-to-TextSTTEmbeddingsEmbeddingsFull leaderboardFull298OpenAll90d30dRankModelLLM StatsReasoningCodingAgentCode ArenaContextSpeedPricing $/MLicense1Claude Mythos PreviewUNRELEASEDAnthropic70.371.357.348.9————Proprietary2GPT-5.5OpenAI64.362.953.144.11,8471.1M28c/s$7.78Proprietary3Claude Opus 4.7Anthropic61.362.751.642.41,9151.0M54c/s$7.22Proprietary4GPT-5.4OpenAI61.358.044.337.91,7251.0M135c/s$3.89Proprietary5GPT-5.2 ProOpenAI61.256.9—29.9————Proprietary6Kimi K2.6Moonshot AI59.059.245.638.81,260262K88c/s$1.29Open Source7Gemini 3.1 ProGoogle58.059.144.133.92,0931.0M—$3.89Proprietary8Claude Opus 4.6Anthropic57.760.045.638.42,0181.0M139c/s$7.22Proprietary9Seed 2.0 ProByteDance57.054.733.329.3————Proprietary10Gemini 3 ProGoogle56.550.233.424.21,579———Proprietary11GPT-5.2OpenAI56.354.135.726.51,514400K200c/s$3.11Proprietary12GPT-5.1 ThinkingOpenAI54.947.930.8—1,024400K31c/s$2.22Proprietary13Gemini 3 FlashGoogle54.649.631.525.61,6961.0M482c/s$0.78Proprietary14GPT-5.1OpenAI54.347.831.8—1,160400K540c/s$2.22Proprietary15GPT-5.1 HighOpenAI53.653.8——1,140———Proprietary16Muse SparkMeta53.252.932.925.1————Proprietary17Qwen3.6 PlusAlibaba Cloud / Qwen Team52.253.143.332.01,0241.0M94c/s$0.78Proprietary18GPT-5.1 InstantOpenAI52.148.731.3—814400K382c/s$2.22Proprietary19GPT-5 MediumOpenAI52.043.9——1,101—95c/s—Proprietary20DeepSeek-V4-Pro-MaxDeepSeek52.057.845.036.99161.0M79c/s$1.93Open Source1-20 of 298Previous12345NextRecentNew ModelsAnnounced in the last 15 days.All updatesIndexPerformance IndexComposite TrueSkill ratings across published benchmarks.Full leaderboardLeaderboard guideCompare the best AI models with one independent score.The LLM Stats leaderboard ranks GPT, Claude, Gemini, Llama, DeepSeek, Qwen, Mistral, GLM and more by intelligence, speed and price. Every score is sourced from public benchmarks and live API metrics.Read the methodologyHow the LLM Stats Score is computedCompare two modelsPricing, context, speed and benchmark scoresExplore benchmarksMMLU, GPQA, SWE-Bench, AIME and moreFAQQuick answers for choosing, comparing and interpreting today's leading AI models.Which AI model ranks #1 on the LLM Leaderboard?On the LLM Stats Leaderboard, Claude Mythos Preview currently leads on GPQA Diamond (94.6% gpqa), the most discriminating reasoning benchmark at the frontier. This AI leaderboard ranks models by the LLM Stats Score, which aggregates GPQA, SWE-Bench Verified, coding-arena performance and pricing into one comparable AI ranking. Rankings refresh continuously as new benchmark results land.What is the best AI model right now?"Best" depends on what you're optimizing for. For frontier reasoning, Claude Mythos Preview leads on GPQA. For coding agents, Gemini 3.1 Pro is the strongest in head-to-head coding-arena play. For low cost at frontier quality, Kimi K2.6 is the cheapest in the top 10 at $0.95 /M tok. The leaders summary above the table names the current winner per axis.What are the best LLMs in 2026?The leading LLMs in 2026 are Claude Mythos Preview, Gemini 3.1 Pro, and the frontier models from OpenAI (GPT-5 family), Anthropic (Claude Opus and Sonnet), Google (Gemini 3 Pro), xAI (Grok 4), DeepSeek (V3 / R1) and Z.AI (GLM-5). Open-weights leaders include Llama, Qwen and DeepSeek. The full ranking is in the leaderboard table above.What is the cheapest AI model in the top 10?Kimi K2.6 is the cheapest model in the top 10 by GPQA Diamond, at $0.95 /M tok input. The Cheapest filter on the leaderboard restricts to verified, currently-available frontier models — pricing is pulled from each provider's public price list and cross-checked against billing samples through the LLM Stats proxy.Which AI model has the largest context window?Grok-4.20 Beta Non-Reasoning currently exposes the largest practical context window at 2.0M tokens tokens. Larger context lets you keep more documents, conversation history and tool traces in a single request. For long-document workloads, also consult the per-model "effective context" notes on each model detail page — providers vary in how well they actually use the upper end of their advertised windows.What is the fastest LLM by output speed?Mercury 2 currently has the highest output throughput at 1197 tok/s. Output speed is measured by routing standardized prompts through each provider's API and averaging tokens-per-second over a 7-day rolling window. Fast inference matters most for streaming chat UIs and agentic loops; for batched async workloads, blended price per 1M tokens is usually the better axis.Which is the best open-source AI model?Kimi K2.6 currently leads among open-weights LLMs (90.5% gpqa). The open-weights ecosystem is dominated by Llama, Qwen, DeepSeek, Mistral, GLM and Gemma. The dedicated Open LLM Leaderboard filters this catalog to models with publicly released weights so you can self-host or fine-tune.How is the LLM Stats Score calculated?The LLM Stats Score is a composite that blends verified benchmark results (GPQA Diamond, SWE-Bench Verified, coding-arena), live performance metrics (output throughput, time-to-first-token) and per-token pricing into one comparable number. Pricing and metadata revalidate hourly; live performance updates on a 7-day rolling average. For the full weighting and refresh cadence see the LLM Stats Score methodology. 298 canonical models are tracked across every major lab and inference provider.

### Error handling for MCP Servers

Use interceptors to catch tool execution errors and implement retry logic:

In [17]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."


### Creating an Agent with MCP Client Tools and error handling

In [18]:
# Connect to the remote MCP server over Streamable HTTP
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    },
    tool_interceptors=[retry_interceptor, fallback_interceptor]
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
# model.max_completion_tokens=131072
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [31]:
from langchain_core.messages import HumanMessage

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="""

You are a web research assistant.

Goal: find the latest 2026 news about AI Agents. Search for 2 results and fetch each page.

Default behavior:
Start with the smallest useful action.
Prefer the shortest path that can answer the request correctly.
Do not search multiple engines by default.
Do not fetch full pages unless the answer needs more detail than search snippets provide.
Do not fetch many pages for a simple factual answer; by default, deepen only the top 1-2 most relevant results.
Stop once the available evidence is enough to answer the user correctly.
Expand the search only when the first pass is insufficient, ambiguous, or clearly low quality.
maxChars for fetchWebContent should be 200000 by default, but can be increased for complex topics.
If any fetch result fqiledm skip that result and continue with others, do not fail the entire request.
Exclude results from reddit.com.

Decision rules:
First priority: if the user gives a specific public URL, fetch that URL directly instead of searching first.
Second priority: if the user asks for current information, broad discovery, or comparisons, start with a single focused search.
Third priority: if a search result looks promising but the snippet is insufficient, use fetchWebContent on that result URL.
Repository priority: if the target is a GitHub repository, prefer fetchGithubReadme over generic page fetching.
Escalation rule: only move to multi-engine cross-checking when one focused pass is insufficient.

Engine selection:
Prefer startpage for general English-language web search when it is available.
Use bing as a secondary broad web engine when needed. If request-mode Bing is blocked, suggest SEARCH_MODE=auto.
If Bing Playwright mode returns no results for a site:-restricted query, retry once without the site: prefix before concluding the target has no usable results.
Use baidu, csdn, or juejin when the user clearly wants Chinese-language or China-hosted sources.
Treat engine choice as a heuristic, not a hard rule. If a preferred engine is unavailable or poor quality, switch.
Use multiple engines only when cross-checking is useful. Do not add engines just for variety.

""")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================



You are a web research assistant.

Goal: find the latest 2026 news about AI Agents. Search for 2 results and fetch each page.

Default behavior:
Start with the smallest useful action.
Prefer the shortest path that can answer the request correctly.
Do not search multiple engines by default.
Do not fetch full pages unless the answer needs more detail than search snippets provide.
Do not fetch many pages for a simple factual answer; by default, deepen only the top 1-2 most relevant results.
Stop once the available evidence is enough to answer the user correctly.
Expand the search only when the first pass is insufficient, ambiguous, or clearly low quality.
maxChars for fetchWebContent should be 200000 by default, but can be increased for complex topics.
If any fetch result fqiledm skip that result and continue with others, do not fail the entire request.
Exclude results from reddit.com.

Decision rules:
Fir

In [27]:
# Fetch the page content using the fetch tool
result = await fetch_web_content_tool.ainvoke({"url": "https://www.reddit.com/r/automation/comments/1s73adp/my_favorite_ai_agents_in_2026_sorted_by_use_case/", "maxChars": 30000})

# Parse and display the fetched content
web_content_json = json.loads(result[0]["text"])
print(json.dumps(web_content_json, indent=2))

ToolException: Failed to fetch web content: No readable content was extracted from this URL

In [ ]:
import os
from typing import Annotated, Literal

import httpx
from langchain.tools import InjectedToolArg, tool
from markdownify import markdownify

@tool
def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch webpage and convert HTML to markdown."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching {url}: {e!s}"

In [32]:
fetch_webpage_content("https://blog.cloudflare.com/agents-week-in-review/")

"  Building the agentic cloud: everything we launched during Agents Week 2026\n\n[Get Started Free](https://dash.cloudflare.com/sign-up)|[Contact Sales](https://www.cloudflare.com/plans/enterprise/contact/)|\n\n▼\n\n[![The Cloudflare Blog](https://cf-assets.www.cloudflare.com/zkvhlag99gkb/69RwBidpiEHCDZ9rFVVk7T/092507edbed698420b89658e5a6d5105/CF_logo_stacked_blktype.png)](/)\n\n[The Cloudflare Blog](/)\n------------------------\n\nSubscribe to receive notifications of new posts:\n\nSubscribe\n\n![magnifier icon](/images/magnifier.svg)![hamburger menu](/images/hamburger.svg)\n\n[AI](/tag/ai/)\n\n[Developers](/tag/developers/)\n\n[Radar](/tag/cloudflare-radar/)\n\n[Product News](/tag/product-news/)\n\n[Security](/tag/security/)\n\n[Policy & Legal](/tag/policy/)\n\n[Zero Trust](/tag/zero-trust/)\n\n[Speed & Reliability](/tag/speed-and-reliability/)\n\n[Life at Cloudflare](/tag/life-at-cloudflare/)\n\n[Partners](/tag/partners/)\n\n[AI](/tag/ai/)\n\n[Developers](/tag/developers/)\n\n[Radar

In [39]:
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

agent_with_mcp = create_agent(model=model, tools=[search_tool, fetch_webpage_content])

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="""
    What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    Search the web for about 20 pages.
    """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================


    What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    Search the web for about 20 pages.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_TldCwFpgWfDVeLXhwUuvV7Ow)
 Call ID: call_TldCwFpgWfDVeLXhwUuvV7Ow
  Args:
    query: latest news updates AI agents Microsoft Google AWS Anthropic OpenAI site:news OR blog 2025
    limit: 20
    searchMode: request
    engines: ['startpage', 'duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest news updates AI agents Microsoft Google AWS Anthropic OpenAI site:news OR blog 2025",\n  "engines": [\n    "startpage",\n    "duckduckgo"\n  ],\n  "totalResults": 10,\n  "results": [\n    {\n      "title": "AI News: OpenAI Splits from Microsoft, Anthropic Step